In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import ast
from sklearn.model_selection import KFold
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from torchmetrics import Accuracy

#### Mitochondrial Movement Dectection CNN

In [ ]:
# edit path for desired experiment
experiment = ''

# cytosolic Ca channel
_, cyt_data = cv2.imreadmulti(f"ca_force_movies/{experiment}/561_registered.tif", flags=cv2.IMREAD_UNCHANGED)
# traction force channel
_, tract_data = cv2.imreadmulti(f"ca_force_movies/{experiment}/traction_maps.tif", flags=cv2.IMREAD_UNCHANGED)
# mitochondrial Ca channel
_, mit_data = cv2.imreadmulti(f"ca_force_movies/{experiment}/488_registered.tif", flags=cv2.IMREAD_UNCHANGED)

In [3]:
# read in rois
rois = pd.read_csv(f'{experiment}_rois.csv')

rois["dim"] = rois["dim"].apply(ast.literal_eval)
rois["centroids"] = rois["centroids"].apply(ast.literal_eval)

rois = rois.to_dict(orient='records')

##### Labeling and Preparing the Data

Hand-labeled frames based on movment (0 - no movement, 1 - active)

In [4]:
# adjust labels to specific cells/ROIs
mit_labels = [1,1,1,1,1,0,0,0,0,0,1,0]

Pull motion from the mitochondrial channel

In [5]:
# mit Ca channel motion masks
mit_motion = []

for frame_idx in range(1, len(mit_data)):

    # collect prev and curr frames
    prev_frame = mit_data[frame_idx-1]
    curr_frame = mit_data[frame_idx]

    # find differences between frames (pixel absolute difference)
    frame_diff = cv2.absdiff(curr_frame, prev_frame)

    # append to list
    mit_motion.append(frame_diff)

Stack all 3 channels (cyt, tract, mit)

In [ ]:
X = []
Y = []
roi_ids = []

for frame_idx in range(1, len(cyt_data)):

    for roi_idx, roi in enumerate(rois):

        xmin, xmax, ymin, ymax = roi["dim"]

        # extract the ROI from all 3 channels
        cyt_patch = cyt_data[frame_idx][ymin:ymax, xmin:xmax]
        tract_patch = tract_data[frame_idx][ymin:ymax, xmin:xmax]
        mit_patch = mit_motion[frame_idx-1][ymin:ymax, xmin:xmax] 

        # stack all 3 channels for current ROI
        patch = np.stack([cyt_patch, tract_patch, mit_patch], axis=0)

        # add stacked ROI to X, assign ROI label to Y, 
        X.append(patch)
        Y.append(mit_labels[roi_idx])
        roi_ids.append(roi_idx)

X = np.array(X)
Y = np.array(Y)

# check input and label shape
print(X.shape)
print(Y.shape)

Normalize the 3 channels

In [7]:
X = X.astype(np.float32)

for c in range(3):

    mean = X[:,c].mean()
    std = X[:,c].std()
    X[:,c] = (X[:,c] - mean) / std

Shuffle ROIs as either train or test set

In [ ]:
# set seeds to desired value
rng = np.random.default_rng(145)

all_rois = np.unique(roi_ids)
rng.shuffle(all_rois)

train_rois = set(all_rois[:int(0.7*len(all_rois))])
test_rois = set(all_rois[int(0.7*len(all_rois)):])

Convert stacked data/mit labels into a torch dataset, train-test split and data loader

In [9]:
torch.manual_seed(145)

# convert to tensor
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.long)

# initialize dataset
class MitDataset(Dataset):

    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

    def __len__(self):
            return len(self.Y)

dataset = MitDataset(X, Y)

# extract indices for train and test ROIs
train_indices = [i for i, r in enumerate(roi_ids) if r in train_rois]
test_indices  = [i for i, r in enumerate(roi_ids) if r in test_rois]

# subset the data and convert to DataLoader
train_set = torch.utils.data.Subset(dataset, train_indices)
test_set  = torch.utils.data.Subset(dataset, test_indices)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = DataLoader(test_set, batch_size=16, shuffle=True)

##### Training the CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1) 
        x = self.fc1(x)           
        return x

model = CNN(in_channels=3, num_classes=2)

# initialize loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# train model over 10 epochs
epochs=10
for epoch in range(epochs):
 # iterate over training batches
   print(f"Epoch [{epoch + 1}/{epochs}]")

   for batch_index, (data, targets) in enumerate(tqdm(train_loader)):
       optimizer.zero_grad()

       scores = model(data)
       loss = criterion(scores, targets)
       
       loss.backward()
       optimizer.step()

Model accuracy

In [ ]:
# set binary accuracy metric (1 moving 0 not moving)
acc = Accuracy(task="binary")

model.eval()
with torch.no_grad():
   for images, labels in test_loader:
       # get predictions for test set
       outputs = model(images)
       _, preds = torch.max(outputs, 1)
       acc(preds, labels) 

# test accuracy
test_accuracy = acc.compute()
print(f"Test accuracy: {test_accuracy}")

##### Gradient-weighted Class Activation Mapping (saliency maps)

In [ ]:
# get activations and gradients from the last convolutional layer of the model
activations = None
gradients = None

def forward_hook(module, input, output):
    global activations
    activations = output.detach()

def full_backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0].detach()

model.conv3.register_forward_hook(forward_hook)
model.conv3.register_full_backward_hook(full_backward_hook)


def compute_heatmap(model, img):
    model.eval()

    # add batch dimension
    img = img.unsqueeze(0)
    img.requires_grad = True

    # compute logits from the model
    logits = model(img)
    model.zero_grad()

    # model's prediction 
    pred = logits.max(-1)[-1].item()

    # compute gradients with respect to the model's most confident prediction
    logits[0, pred].backward(retain_graph=True)

    # average gradients of the featuremap
    pool_grads = gradients.mean(dim=[2,3], keepdim=True)

    # multiply each activation map with corresponding gradient average
    heatmap = (activations * pool_grads).mean(dim=1)

    heatmap = heatmap.squeeze().cpu().numpy()

    return heatmap, pred

In [13]:
def upsample_heatmap(map, image):
    # permute image
    image = image.squeeze(0).permute(1, 2, 0).cpu().numpy()

    # normalize the image
    image = (image - image.min())/(image.max()-image.min())

    # normalize the heatmap
    map -= map.min()
    map /= map.max()
    map = np.uint8(255*map)

    # resize the heatmap to the same as the input
    map = cv2.resize(map, (image.shape[1], image.shape[0]))
    map = cv2.applyColorMap(map, cv2.COLORMAP_JET)
    map = cv2.cvtColor(map, cv2.COLOR_BGR2RGB)
    map = np.uint8(map)
    
    # change this to balance between heatmap and image
    map = (map/255.)*0.6 + image*0.4

    return map

Visualize activation maps with each channel

In [ ]:
for roi_idx in range(len(rois)):

    atp_added = 30

    dataset_index = roi_idx + atp_added * len(rois)

    sample_img = X[dataset_index]
    label = Y[dataset_index]

    map, pred = compute_heatmap(model, sample_img)
    overlay = upsample_heatmap(map, sample_img)

    fig,ax = plt.subplots(1,4,figsize=(16,4))

    titles = [f"ROI {roi_idx+1}: Cytosolic Ca", "Traction Force", "Mito Ca"]

    # iterate through each channel
    for i in range(3):
        ax[i].imshow(sample_img[i], cmap='gray')
        ax[i].set_title(titles[i])
        ax[i].axis('off')

    # show heatmap with predicted and true label (1 or 0)
    ax[3].imshow(overlay)
    ax[3].set_title(f"Grad-CAM: Pred={pred}, True={label}")
    ax[3].axis('off')

    plt.show()